# HW04

## 2. 序列模型

### 2.1 理论计算题

字符序列为 `ababc`，相邻字符之间的转移依次为

$$
a\to b,\quad b\to a,\quad a\to b,\quad b\to c.
$$

因此，以 `b` 为前一个字符时，各转移计数为

$$
C(b,a)=1,\qquad C(b,b)=0,\qquad C(b,c)=1,
$$

而从 `b` 出发的总转移数为 $C(b)=2$。词汇表大小 $|\mathcal V|=3$。使用加 1 平滑时，

$$
p(x_t=w\mid x_{t-1}=b)
=\frac{C(b,w)+1}{C(b)+|\mathcal V|}.
$$

所以

$$
p(a\mid b)=\frac{1+1}{2+3}=\boxed{\frac25=0.4},
$$

$$
p(c\mid b)=\frac{1+1}{2+3}=\boxed{\frac25=0.4}.
$$

未出现的转移 $b\to b$ 也被计入平滑，其概率为 $1/5$，三个条件概率之和为 1。

### 2.2 编程题：文本预处理与滑动窗口

下面的函数完成小写转换、标点清理、分词、按频率构造词汇表，以及长度为 $n$ 的滑动窗口生成。
同频词按首次出现顺序分配 ID，使结果具有确定性。为与题目示例一致，最后一个没有后续词的窗口仍保留，标签记为 `None`；用于训练时可以将它过滤掉。

In [1]:
import math
import random
import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


def preprocess_text(text, n):
    """Return vocabulary and (length-n feature windows, next-word labels)."""
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if not isinstance(n, int) or isinstance(n, bool) or n <= 0:
        raise ValueError("n must be a positive integer")

    # 仅保留英文字母和空白；split() 会合并连续空白。
    cleaned_text = re.sub(r"[^a-z\s]", "", text.lower())
    words = cleaned_text.split()

    counts = Counter(words)
    first_position = {word: i for i, word in enumerate(words)}
    sorted_words = sorted(
        counts,
        key=lambda word: (-counts[word], first_position[word]),
    )
    vocabulary = {word: index for index, word in enumerate(sorted_words)}

    features = []
    labels = []
    for start in range(max(0, len(words) - n + 1)):
        end = start + n
        features.append(words[start:end])
        labels.append(words[end] if end < len(words) else None)

    return vocabulary, (features, labels)


demo_text = "The time machine, the TIME!"
vocabulary, (features, labels) = preprocess_text(demo_text, n=2)

print("original text:", demo_text)
print("vocabulary:", vocabulary)
print("features:", features)
print("labels:", labels)

example_vocab, (example_features, example_labels) = preprocess_text(
    "The time machine", n=2
)
assert example_features == [["the", "time"], ["time", "machine"]]
assert example_labels == ["machine", None]
assert list(vocabulary) == ["the", "time", "machine"]
print("preprocess_text checks passed")

original text: The time machine, the TIME!
vocabulary: {'the': 0, 'time': 1, 'machine': 2}
features: [['the', 'time'], ['time', 'machine'], ['machine', 'the'], ['the', 'time']]
labels: ['machine', 'the', 'time', None]
preprocess_text checks passed


---

## 3. 循环神经网络

### 3.1 理论计算题

记

$$
e_t=o_t-y_t,\qquad
h_t=W_{hh}h_{t-1}+W_{hx}x_t,\qquad
o_t=W_{oh}h_t,
$$

且

$$
L=\frac12\sum_{t=1}^{T}\|o_t-y_t\|_2^2.
$$

因为同一个 $W_{hh}$ 在全部时间步共享，来自当前及未来时刻的损失都会通过隐藏状态传回。定义

$$
\delta_t=\frac{\partial L}{\partial h_t},
$$

则 BPTT 递推为

$$
\delta_t=W_{oh}^{\mathsf T}e_t+W_{hh}^{\mathsf T}\delta_{t+1},
\qquad \delta_{T+1}=0.
$$

展开递推可得

$$
\delta_t
=\sum_{k=t}^{T}
\left(W_{hh}^{\mathsf T}\right)^{k-t}
W_{oh}^{\mathsf T}e_k.
$$

在时间步 $t$，$h_t$ 对 $W_{hh}$ 的局部贡献为 $\delta_t h_{t-1}^{\mathsf T}$，因此

$$
\boxed{
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}\delta_t h_{t-1}^{\mathsf T}
=\sum_{t=1}^{T}\sum_{k=t}^{T}
\left(W_{hh}^{\mathsf T}\right)^{k-t}
W_{oh}^{\mathsf T}(o_k-y_k)h_{t-1}^{\mathsf T}
}.
$$

跨越 $k-t$ 个时间步的梯度包含 $\left(W_{hh}^{\mathsf T}\right)^{k-t}$：

- 若其主导奇异值（长期行为也可由谱半径判断）小于 1，连乘项指数衰减，产生梯度消失；
- 若主导奇异值大于 1，连乘项可能指数增长，产生梯度爆炸；
- 接近 1 时更容易保持梯度，但具体结果还取决于矩阵方向、误差信号和序列长度。

这里是线性 RNN；若带激活函数，还需要把每一步激活函数的 Jacobian 一起连乘。

### 3.2 编程题：RNN 单元的前向与单步反向传播

采用批量优先的矩阵约定：$W_{hx}\in\mathbb R^{D\times H}$，$W_{hh}\in\mathbb R^{H\times H}$，

$$
h_t=\tanh(x_tW_{hx}+h_{t-1}W_{hh}+b_h).
$$

反向传播由 $\mathrm d a=\mathrm d h_t\odot(1-h_t^2)$ 得到，其结果再与 PyTorch 自动微分逐项核对。

In [2]:
def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    pre_activation = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = torch.tanh(pre_activation)
    cache = (x_t, h_prev, W_hx, W_hh, h_t)
    return h_t, cache


def rnn_step_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, h_t = cache
    da = dh_next * (1.0 - h_t.square())

    dx_t = da @ W_hx.T
    dh_prev = da @ W_hh.T
    dW_hx = x_t.T @ da
    dW_hh = h_prev.T @ da
    db_h = da.sum(dim=0)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


batch_size, input_size, hidden_size = 2, 3, 4
x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)
W_hx = torch.randn(input_size, hidden_size)
W_hh = torch.randn(hidden_size, hidden_size)
b_h = torch.randn(hidden_size)
dh_next = torch.randn(batch_size, hidden_size)

h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
manual_grads = rnn_step_backward(dh_next, cache)
gradient_names = ["dx_t", "dh_prev", "dW_hx", "dW_hh", "db_h"]

# 使用相同数据建立自动微分计算图，验证手动梯度。
x_auto = x_t.clone().requires_grad_()
h_auto_prev = h_prev.clone().requires_grad_()
W_hx_auto = W_hx.clone().requires_grad_()
W_hh_auto = W_hh.clone().requires_grad_()
b_auto = b_h.clone().requires_grad_()
h_auto = torch.tanh(
    x_auto @ W_hx_auto + h_auto_prev @ W_hh_auto + b_auto
)
h_auto.backward(dh_next)
autograd_grads = (
    x_auto.grad,
    h_auto_prev.grad,
    W_hx_auto.grad,
    W_hh_auto.grad,
    b_auto.grad,
)

print("h_t shape:", tuple(h_t.shape))
for name, manual_grad, auto_grad in zip(
    gradient_names, manual_grads, autograd_grads
):
    max_error = (manual_grad - auto_grad).abs().max().item()
    print(f"{name:7s} shape={tuple(manual_grad.shape)}, max error={max_error:.3e}")
    assert torch.allclose(manual_grad, auto_grad, atol=1e-6)
print("RNN forward/backward checks passed")

h_t shape: (2, 4)
dx_t    shape=(2, 3), max error=0.000e+00
dh_prev shape=(2, 4), max error=0.000e+00
dW_hx   shape=(3, 4), max error=0.000e+00
dW_hh   shape=(4, 4), max error=0.000e+00
db_h    shape=(4,), max error=0.000e+00
RNN forward/backward checks passed


---

## 4. 高级循环神经网络

### 4.1 理论计算题

采用标准 RNN 形式：每个方向、每一层只有一个合并后的隐藏偏置向量。第 1 层每个方向的参数量为

$$
HD+H^2+H.
$$

从第 2 层开始，输入是上一层两个方向的拼接结果，维度为 $2H$，所以每层每个方向的参数量为

$$
H(2H)+H^2+H=3H^2+H.
$$

双向 RNN 主体共有

$$
2\left[HD+H^2+H+(L-1)(3H^2+H)\right]
$$

个参数。最后输出层从 $2H$ 维映射到 $O$ 维，包含

$$
2HO+O
$$

个参数。因此总参数量为

$$
\boxed{
N_{mathrm{total}}
=2\left[HD+H^2+H+(L-1)(3H^2+H)\right]+2HO+O
}.
$$

若使用 PyTorch `nn.RNN` 的默认实现，每层每个方向分别保存 `bias_ih` 和 `bias_hh` 两个偏置；相对于上述合并偏置约定，需要再增加 $2LH$ 个参数。

### 4.2 编程题：双向 RNN 编码器

`nn.RNN` 的逐时刻输出已经按 `[前向, 后向]` 拼接。序列表示应由前向网络处理完最后一个词后的状态与后向网络处理完整个反向序列后的状态拼接，即使用 `h_n`，而不是简单取 `output[-1]` 的全部通道。

In [3]:
class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True,
        )

    def forward(self, X):
        # output: (seq_len, batch, 2 * hidden_dim)
        # h_n: (2, batch, hidden_dim), 顺序为前向、后向。
        output, h_n = self.rnn(X)
        sequence_representation = torch.cat((h_n[0], h_n[1]), dim=-1)
        return output, sequence_representation


seq_len, batch_size, input_dim, hidden_dim = 6, 3, 5, 4
X_sequence = torch.randn(seq_len, batch_size, input_dim)
encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
all_hidden_states, sequence_representation = encoder(X_sequence)

print("input shape:                 ", tuple(X_sequence.shape))
print("all hidden states shape:     ", tuple(all_hidden_states.shape))
print("sequence representation shape:", tuple(sequence_representation.shape))

assert all_hidden_states.shape == (seq_len, batch_size, 2 * hidden_dim)
assert sequence_representation.shape == (batch_size, 2 * hidden_dim)
# 最后前向状态和最后后向状态分别也出现在 output 的两端。
assert torch.allclose(
    sequence_representation[:, :hidden_dim],
    all_hidden_states[-1, :, :hidden_dim],
)
assert torch.allclose(
    sequence_representation[:, hidden_dim:],
    all_hidden_states[0, :, hidden_dim:],
)
print("bidirectional RNN encoder checks passed")

input shape:                  (6, 3, 5)
all hidden states shape:      (6, 3, 8)
sequence representation shape: (3, 8)
bidirectional RNN encoder checks passed


---

## 5. 嵌入向量

### 5.1 理论计算题

给定中心词 $w_c$、真实上下文词 $w_o$，中心词输入向量为 $v_c$，输出词向量为 $u_o$。从噪声分布中独立采样 $K$ 个负样本 $n_1,\ldots,n_K$，对应输出向量为 $u_{n_k}$。一个正样本及其负样本的对数似然目标为

$$
J_{mathrm{NS}}
=\log\sigma(u_o^{\mathsf T}v_c)
+\sum_{k=1}^{K}\log\sigma(-u_{n_k}^{\mathsf T}v_c),
$$

其中 $\sigma(z)=1/(1+e^{-z})$。训练时最小化其负值：

$$
\boxed{
\mathcal L_{mathrm{NS}}
=-\log\sigma(u_o^{\mathsf T}v_c)
-\sum_{k=1}^{K}\log\sigma(-u_{n_k}^{\mathsf T}v_c)
}.
$$

负样本通常从基于词频的噪声分布独立采样：

$$
P_n(w)=\frac{f(w)^{3/4}}{\sum_{w'\in\mathcal V}f(w')^{3/4}},
$$

其中 $f(w)$ 是词频。$3/4$ 次幂会相对降低高频词的支配程度，同时又不会像均匀采样那样过度采到罕见词。实际实现一般排除本次真实上下文词；是否允许负样本重复取决于采样实现。

### 5.2 编程题：完整 Softmax 的 CBOW 损失

上下文索引形状为 `(batch_size, context_size)`。先查表并沿上下文维求平均，再乘输出权重得到所有词的 logits，最后通过 `logsumexp` 稳定地计算完整 softmax 交叉熵。

In [4]:
def cbow_loss(context_indices, target_indices, W, W_out):
    if context_indices.dim() != 2:
        raise ValueError("context_indices must have shape (batch, context_size)")
    if target_indices.dim() != 1 or target_indices.shape[0] != context_indices.shape[0]:
        raise ValueError("target_indices must have shape (batch,)")
    if W.dim() != 2 or W_out.dim() != 2 or W.shape[1] != W_out.shape[0]:
        raise ValueError("W and W_out have incompatible shapes")

    context_vectors = W[context_indices]                 # (B, C, d)
    hidden = context_vectors.mean(dim=1)                 # (B, d)
    logits = hidden @ W_out                              # (B, V)
    log_probs = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    per_sample_loss = -log_probs.gather(1, target_indices[:, None]).squeeze(1)
    return per_sample_loss.mean()


V, d = 7, 4
context_indices = torch.tensor([[0, 2, 3, 1], [4, 1, 5, 2]])
target_indices = torch.tensor([4, 0])
W = (0.1 * torch.randn(V, d)).requires_grad_()
W_out = (0.1 * torch.randn(d, V)).requires_grad_()

loss = cbow_loss(context_indices, target_indices, W, W_out)
loss.backward()

with torch.no_grad():
    hidden = W[context_indices].mean(dim=1)
    probabilities = torch.softmax(hidden @ W_out, dim=1)

print("context shape:", tuple(context_indices.shape))
print("probability shape:", tuple(probabilities.shape))
print("probability row sums:", probabilities.sum(dim=1))
print(f"CBOW cross-entropy loss: {loss.item():.6f}")
print("W gradient shape:", tuple(W.grad.shape))
print("W_out gradient shape:", tuple(W_out.grad.shape))

assert probabilities.shape == (context_indices.shape[0], V)
assert torch.allclose(probabilities.sum(dim=1), torch.ones(2), atol=1e-6)
assert torch.isfinite(loss)
print("CBOW loss checks passed")

context shape: (2, 4)
probability shape: (2, 7)
probability row sums: tensor([1., 1.])
CBOW cross-entropy loss: 1.941743
W gradient shape: (7, 4)
W_out gradient shape: (4, 7)
CBOW loss checks passed


---

## 6. 注意力机制

### 6.1 理论计算题

题目只给出了 $Q,K,V$ 的维度而没有给出具体元素，因此用符号写出完整数值计算流程。由于 $d_k=4$，缩放因子为 $\sqrt{d_k}=2$。

第一步，计算得分矩阵

$$
S=\frac{QK^{\mathsf T}}{2}\in\mathbb R^{2\times3},
\qquad
s_{ij}=\frac12\sum_{r=1}^{4}q_{ir}k_{jr}.
$$

展开形状为

$$
S=\frac12
\begin{bmatrix}
q_1\cdot k_1&q_1\cdot k_2&q_1\cdot k_3\\
q_2\cdot k_1&q_2\cdot k_2&q_2\cdot k_3
\end{bmatrix}.
$$

第二步，对每一行的 3 个键做 softmax：

$$
A=\operatorname{softmax}(S),\qquad
a_{ij}=\frac{e^{s_{ij}}}{\sum_{\ell=1}^{3}e^{s_{i\ell}}},
\qquad A\in\mathbb R^{2\times3}.
$$

即

$$
A=
\begin{bmatrix}
a_{11}&a_{12}&a_{13}\\
a_{21}&a_{22}&a_{23}
\end{bmatrix},
\qquad
\sum_{j=1}^{3}a_{ij}=1.
$$

第三步，对值向量加权求和：

$$
\boxed{O=AV\in\mathbb R^{2\times5}},
$$

$$
o_{im}=\sum_{j=1}^{3}a_{ij}v_{jm},
$$

或按行写成

$$
O=
\begin{bmatrix}
a_{11}v_1+a_{12}v_2+a_{13}v_3\\
a_{21}v_1+a_{22}v_2+a_{23}v_3
\end{bmatrix}.
$$

因此最终输出包含 2 个查询对应的 5 维加权值向量。

### 6.2 编程题：多头注意力前向传播

下面从线性投影和张量重排开始实现两头注意力，不调用 `nn.MultiheadAttention`。每个头的维度为 $d_k=d_v=4/2=2$。

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attention_weights = None

    def _split_heads(self, tensor):
        # (S, B, d_model) -> (B, num_heads, S, head_dim)
        seq_len, batch_size, _ = tensor.shape
        tensor = tensor.permute(1, 0, 2)
        tensor = tensor.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        return tensor.permute(0, 2, 1, 3)

    def forward(self, X):
        if X.dim() != 3 or X.shape[-1] != self.d_model:
            raise ValueError("X must have shape (seq_len, batch, d_model)")

        Q = self._split_heads(self.W_q(X))
        K = self._split_heads(self.W_k(X))
        V = self._split_heads(self.W_v(X))

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        attention = torch.softmax(scores, dim=-1)
        self.attention_weights = attention.detach()
        context = attention @ V

        # (B, heads, S, head_dim) -> (S, B, d_model)
        batch_size, _, seq_len, _ = context.shape
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.reshape(batch_size, seq_len, self.d_model)
        context = context.permute(1, 0, 2)
        return self.W_o(context)


seq_len, batch_size, d_model = 5, 2, 4
X_attention = torch.randn(seq_len, batch_size, d_model)
mha = MultiHeadAttention(d_model=4, num_heads=2)
attention_output = mha(X_attention)

print("input shape:    ", tuple(X_attention.shape))
print("attention shape:", tuple(mha.attention_weights.shape))
print("output shape:   ", tuple(attention_output.shape))
print("first head, first query weights:")
print(mha.attention_weights[0, 0, 0])
print("attention row sums:")
print(mha.attention_weights.sum(dim=-1))

assert attention_output.shape == X_attention.shape
assert mha.attention_weights.shape == (
    batch_size,
    2,
    seq_len,
    seq_len,
)
assert torch.allclose(
    mha.attention_weights.sum(dim=-1),
    torch.ones(batch_size, 2, seq_len),
    atol=1e-6,
)
print("multi-head attention checks passed")

input shape:     (5, 2, 4)
attention shape: (2, 2, 5, 5)
output shape:    (5, 2, 4)
first head, first query weights:
tensor([0.3456, 0.1357, 0.2565, 0.1043, 0.1580])
attention row sums:
tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]])
multi-head attention checks passed
